In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql import DataFrame
from functools import reduce
from delta.tables import DeltaTable
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime
import pprint
import os
import json
import time

## CVM - Fundos Imobiliarios

### 1. Verificando os arquivos

In [0]:
BASE_URL = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/"

response = requests.get(BASE_URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# regex para pegar apenas arquivos de 2026      
pattern = re.compile(r"inf_mensal_fii_(\d{4})\.zip")

files = []


for link in soup.find_all("a", href=True):
    href = link["href"]
    match = pattern.match(href)
    if match:
        files.append(urljoin(BASE_URL, href))



print(files)

### 2. Extraindo os arquivos

In [0]:
for zip_url in files: 
    print(f'Processando: {zip_url}')

    response = requests.get(zip_url)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            if file_name.endswith(".csv"):
                output_path = os.path.join("/Volumes/workspace/case_spark_cvm/raw/cvm_fii/", file_name)

                with z.open(file_name) as source, open(output_path, "wb") as target:
                    target.write(source.read())

                print(f"  → Extraído: {output_path}")


### 3. Salvar em camada Bronze Particionada

In [0]:
import re
import os
from datetime import datetime
import pyspark.sql.functions as f

# Caminho RAW
RAW_PATH = "/Volumes/workspace/case_spark_cvm/raw/cvm_fii/"

# Data de processamento
data_proc = int(datetime.now().strftime(r"%Y%m%d"))

# Regex para identificar tipo do arquivo
pattern = re.compile(r"inf_mensal_fii_(ativo_passivo|complemento|geral)_\d{4}\.csv")

# Lista arquivos da pasta RAW
arquivos = dbutils.fs.ls(RAW_PATH)

# Dicionário para organizar arquivos por tipo
lista_registros = {
    "cvm_fii_ativo_passivo": [],
    "cvm_fii_complemento": [],
    "cvm_fii_geral": []
}

# Classifica os arquivos
for file in arquivos:
    nome_arquivo = os.path.basename(file.path)
    match = pattern.match(nome_arquivo)

    if match:
        tipo = match.group(1)
        if tipo == "ativo_passivo":
            lista_registros["cvm_fii_ativo_passivo"].append(file.path)
        elif tipo == "complemento":
            lista_registros["cvm_fii_complemento"].append(file.path)
        elif tipo == "geral":
            lista_registros["cvm_fii_geral"].append(file.path)



for name_path, caminhos in lista_registros.items():

    if not caminhos:
        continue

    output_path = f"/Volumes/workspace/case_spark_cvm/bronze/{name_path}/"

    print(f"\n Processamento: {name_path}")
    print(f"Arquivos: {len(caminhos)}")
    print(f"Output: {output_path}")

    # ---------------------------------------------------------
    # SOLUÇÃO PARA O SCHEMA DRIFT NO ARQUIVO GERAL
    # ---------------------------------------------------------
    if name_path == "cvm_fii_geral":
        df_final = None
        
        for path in caminhos:
            df_temp = spark.read.csv(path, sep=';', header=True)
            
            # 1. Renomeia colunas legadas (2016) para o padrão atual
            if "CNPJ_Fundo" in df_temp.columns:
                df_temp = df_temp.withColumnRenamed("CNPJ_Fundo", "CNPJ_FUNDO_CLASSE")
            
            if "Nome_Fundo" in df_temp.columns:
                df_temp = df_temp.withColumnRenamed("Nome_Fundo", "Nome_Fundo_Classe")
            
            # 2. Adiciona a coluna Tipo_Fundo_Classe caso ela não exista
            if "Tipo_Fundo_Classe" not in df_temp.columns:
                df_temp = df_temp.withColumn("Tipo_Fundo_Classe", f.lit(None).cast("string"))

            # 3. Empilha os dataframes mapeando pelo NOME da coluna, não pela posição
            if df_final is None:
                df_final = df_temp
            else:
                # allowMissingColumns=True garante que se houver mais alguma coluna nova, o código não quebre
                df_final = df_final.unionByName(df_temp, allowMissingColumns=True)
        
        df = df_final

    else:
        # Leitura padrão em lote para ativo_passivo e complemento
        df = spark.read.csv(caminhos, sep=';', header=True)
    # ---------------------------------------------------------

    # Adiciona data processamento
    df = df.withColumn(
        "data_processamento",
        f.lit(data_proc)
    )

    # Escrita em Delta
    df.write \
        .mode('overwrite') \
        .option("mergeSchema", "true") \
        .option("replaceWhere", f"data_processamento = {data_proc}") \
        .partitionBy('data_processamento') \
        .format('delta') \
        .save(output_path)

In [0]:
display(df)